In [1]:
import requests
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
response = requests.get(url)
with open('input.txt', 'w', encoding='utf-8') as f:
    f.write(response.text)

print("File saved as input.txt")
print(f"Dataset length: {len(response.text)} characters")

File saved as input.txt
Dataset length: 1115394 characters


In [3]:
chars = sorted(list(set(response.text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [4]:
stoi = {s:i for i, s in enumerate(chars)}
itos = {i:s for s, i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [5]:
# Tokenize the entire dataset 

# Encode 
import torch 
data = torch.tensor(encode(response.text), dtype=torch.long)
print(data.shape, data.dtype)
# print(data[:1000])

torch.Size([1115394]) torch.int64


In [6]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [7]:
block_size = 8 
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [8]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    print(t)
    context = x[:t+1]
    target = y[t]
    print(f"When input is {context}, the target is {target}")
    

0
When input is tensor([18]), the target is 47
1
When input is tensor([18, 47]), the target is 56
2
When input is tensor([18, 47, 56]), the target is 57
3
When input is tensor([18, 47, 56, 57]), the target is 58
4
When input is tensor([18, 47, 56, 57, 58]), the target is 1
5
When input is tensor([18, 47, 56, 57, 58,  1]), the target is 15
6
When input is tensor([18, 47, 56, 57, 58,  1, 15]), the target is 47
7
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), the target is 58


In [9]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences/chunks will we process in parallel? 
block_size = 8 # what the maximum context length for prediction 

def get_batch(split):
    # generate a small batch of data of inputs x and targets y 
    data = train_data if split == "train" else val_data
    ix = torch.randint(0, len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs')
print(x.shape)
print(xb)
print('target')
print(yb.shape)
print(yb)

inputs
torch.Size([8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
target
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [10]:
torch.manual_seed(1337)

B, T, C = 4, 8, 2 # batch, time, channels 

x = torch.rand(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [11]:
# Get the rolling average using loops
xbow = torch.zeros(B, T, C)

for b in range(B):
    for t in range(T):
        x_t_rows_rolling = x[b, :t+1]
        xbow[b,t] = torch.mean(x_t_rows_rolling, 0)

In [12]:
# Get rolling average with lower triangular matrix 
# Version 1
torch.manual_seed(1337)
X = torch.randint(1, 3, (4 ,4)).float()
L = torch.tril(torch.ones(4,4)).float()
L = L / torch.sum(L, 1, keepdim=True)

out = L @ X
out

tensor([[2.0000, 2.0000, 1.0000, 1.0000],
        [2.0000, 2.0000, 1.5000, 1.0000],
        [1.6667, 1.6667, 1.3333, 1.0000],
        [1.7500, 1.5000, 1.2500, 1.0000]])

In [13]:
# Get rolling average with lower triangular matrix 
# Version 2
# for B, T, C 

wei = torch.tril(torch.ones(T, T))
wei = wei / torch.sum(wei, 1, keepdim=True)
xbow2 = wei @ x 

In [14]:
torch.allclose(xbow, xbow2)

True

In [15]:
# Get rolling average using lower triangular matrix and softmax 
# Version 3

import torch.nn.functional as F 

tril = torch.tril(torch.ones(T, T))
wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)


In [16]:
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])